# Smart Building Dataset - ML Classification Pipeline
## Anomaly Detection / Fault Classification
### Dataset: normal_training, normal_testing, faulty_training, faulty_testing

In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')
print('✅ Data libraries loaded')

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline
sns.set_style('whitegrid')
print('✅ Visualization libraries loaded')

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV, StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from imblearn.over_sampling import SMOTE
print('✅ ML libraries loaded')

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report, roc_auc_score, roc_curve
import joblib
from datetime import datetime
print('✅ Evaluation libraries loaded')

In [ ]:
RANDOM_STATE = 42
TEST_SIZE = 0.2
CV_FOLDS = 5
TARGET_ACCURACY = 0.90
plt.rcParams['figure.figsize'] = (14, 6)
print(f'✅ Configuration set - Target: {TARGET_ACCURACY*100}%')

## 1. LOAD & EXPLORE DATASET

In [ ]:
df_normal_train = pd.read_csv('data/normal_training.csv')
df_faulty_train = pd.read_csv('data/faulty_training.csv')
df_normal_test = pd.read_csv('data/normal_testing.csv')
df_faulty_test = pd.read_csv('data/faulty_testing.csv')

df_normal_train['label'] = 0
df_faulty_train['label'] = 1
df_normal_test['label'] = 0
df_faulty_test['label'] = 1

df_train = pd.concat([df_normal_train, df_faulty_train], ignore_index=True)
df_test = pd.concat([df_normal_test, df_faulty_test], ignore_index=True)

print(f'✅ Dataset loaded - Train: {len(df_train)}, Test: {len(df_test)}')

In [ ]:
print(df_train.head())
print(f'\nShape: {df_train.shape}')
print(f'Columns: {df_train.columns.tolist()}')

In [ ]:
print('Dataset Info:')
df_train.info()
print(f'\nMissing: {df_train.isnull().sum().sum()}')
print(f'Duplicates: {df_train.duplicated().sum()}')

In [ ]:
print('Statistical Summary:')
print(df_train.describe())

## 2. DATA ANALYSIS

In [ ]:
class_counts = df_train['label'].value_counts()
print(f'Class Distribution:')
print(f'  Normal: {class_counts[0]} ({class_counts[0]/len(df_train)*100:.1f}%)')
print(f'  Faulty: {class_counts[1]} ({class_counts[1]/len(df_train)*100:.1f}%)')

fig, ax = plt.subplots(figsize=(8, 5))
class_counts.plot(kind='bar', ax=ax, color=['green', 'red'])
ax.set_title('Class Distribution')
ax.set_ylabel('Count')
plt.tight_layout()
plt.savefig('output/class_distribution.png', dpi=300, bbox_inches='tight')
plt.show()

ratio = class_counts[1] / class_counts[0]
print(f'\nImbalance ratio: {1/ratio:.2f}:1 - SMOTE will be applied')

In [ ]:
features = [col for col in df_train.columns if col != 'label']
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for idx, feature in enumerate(features[:6]):
    ax = axes[idx // 3, idx % 3]
    df_train[feature].hist(bins=30, ax=ax, alpha=0.7)
    ax.set_title(f'{feature} Distribution')
plt.tight_layout()
plt.savefig('output/feature_distributions.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
correlation = df_train.corr()
plt.figure(figsize=(12, 10))
sns.heatmap(correlation, annot=False, cmap='coolwarm', center=0)
plt.title('Feature Correlation Matrix')
plt.tight_layout()
plt.savefig('output/correlation_matrix.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
for feature in features[:3]:
    Q1 = df_train[feature].quantile(0.25)
    Q3 = df_train[feature].quantile(0.75)
    IQR = Q3 - Q1
    outliers = df_train[(df_train[feature] < Q1 - 1.5*IQR) | (df_train[feature] > Q3 + 1.5*IQR)]
    print(f'{feature}: {len(outliers)} outliers ({len(outliers)/len(df_train)*100:.2f}%)')

## 3. PREPROCESSING

In [ ]:
y_train = df_train['label'].values
y_test = df_test['label'].values
X_train = df_train.drop('label', axis=1).values
X_test = df_test.drop('label', axis=1).values
feature_names = df_train.drop('label', axis=1).columns.tolist()

print(f'✅ Data prepared - Features: {X_train.shape[1]}, Train: {len(X_train)}, Test: {len(X_test)}')

In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f'✅ Scaling complete (fit on training only - NO DATA LEAKAGE)')
print(f'Train mean ≈ 0: {X_train_scaled.mean(axis=0)[:3]}')
print(f'Train std ≈ 1: {X_train_scaled.std(axis=0)[:3]}')

## 4. SMOTE HANDLING

In [ ]:
smote = SMOTE(random_state=RANDOM_STATE, n_jobs=-1)
X_train_balanced, y_train_balanced = smote.fit_resample(X_train_scaled, y_train)

print(f'Original: {len(y_train)} samples')
print(f'After SMOTE: {len(y_train_balanced)} samples')
print(f'Ratio: {(y_train_balanced == 1).sum() / (y_train_balanced == 0).sum():.3f}')

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
pd.Series(y_train).value_counts().plot(kind='bar', ax=ax[0], color=['green', 'red'])
ax[0].set_title('Before SMOTE')
pd.Series(y_train_balanced).value_counts().plot(kind='bar', ax=ax[1], color=['green', 'red'])
ax[1].set_title('After SMOTE')
plt.tight_layout()
plt.savefig('output/smote_balance.png', dpi=300, bbox_inches='tight')
plt.show()
print('✅ SMOTE applied')

## 5. TRAIN MODELS (9 Algorithms)

In [ ]:
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=RANDOM_STATE),
    'KNN': KNeighborsClassifier(n_neighbors=5),
    'Decision Tree': DecisionTreeClassifier(random_state=RANDOM_STATE),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=RANDOM_STATE, n_jobs=-1),
    'Gradient Boosting': GradientBoostingClassifier(random_state=RANDOM_STATE),
    'XGBoost': XGBClassifier(random_state=RANDOM_STATE, n_jobs=-1, verbosity=0),
    'LightGBM': LGBMClassifier(random_state=RANDOM_STATE, n_jobs=-1, verbose=-1),
    'SVM': SVC(kernel='rbf', random_state=RANDOM_STATE, probability=True),
    'Naive Bayes': GaussianNB()
}

print('✅ 9 models initialized')
trained_models = {}
for name, model in models.items():
    model.fit(X_train_balanced, y_train_balanced)
    trained_models[name] = model
    print(f'  ✅ {name} trained')

print(f'\n✅ All models trained')

In [ ]:
predictions = {}
probabilities = {}

for name, model in trained_models.items():
    y_pred = model.predict(X_test_scaled)
    predictions[name] = y_pred
    if hasattr(model, 'predict_proba'):
        probabilities[name] = model.predict_proba(X_test_scaled)[:, 1]
    else:
        probabilities[name] = model.decision_function(X_test_scaled) if hasattr(model, 'decision_function') else y_pred

print(f'✅ Predictions made for all models')

## 6. MODEL EVALUATION

In [ ]:
results = []
for name in models.keys():
    y_pred = predictions[name]
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, zero_division=0)
    rec = recall_score(y_test, y_pred, zero_division=0)
    f1 = f1_score(y_test, y_pred, zero_division=0)
    try:
        roc = roc_auc_score(y_test, probabilities[name])
    except:
        roc = 0
    results.append({'Model': name, 'Accuracy': acc, 'Precision': prec, 'Recall': rec, 'F1': f1, 'ROC-AUC': roc})

results_df = pd.DataFrame(results).sort_values('Accuracy', ascending=False)
print('\n=== MODEL COMPARISON ===')
print(results_df.to_string(index=False))
results_df.to_csv('output/model_results.csv', index=False)
print('\n✅ Results saved')

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 8))
for idx, metric in enumerate(['Accuracy', 'Precision', 'Recall', 'F1', 'ROC-AUC']):
    ax = axes[idx // 3, idx % 3]
    ax.barh(results_df['Model'], results_df[metric], color=plt.cm.Set3(range(len(results_df))))
    ax.set_xlabel(metric)
    ax.axvline(TARGET_ACCURACY, color='red', linestyle='--', linewidth=2, label='Target')
    ax.set_xlim(0, 1.05)
    ax.legend()

axes[1, 2].axis('off')
plt.tight_layout()
plt.savefig('output/metrics_comparison.png', dpi=300, bbox_inches='tight')
plt.show()
print('✅ Metrics visualized')

In [ ]:
best_model_name = results_df.iloc[0]['Model']
best_accuracy = results_df.iloc[0]['Accuracy']
print(f'\n⭐ BEST MODEL: {best_model_name}')
print(f'\nAccuracy: {best_accuracy:.4f} ({best_accuracy*100:.2f}%)')
if best_accuracy >= TARGET_ACCURACY:
    print(f'✅ TARGET ACHIEVED! ({best_accuracy*100:.2f}% >= {TARGET_ACCURACY*100}%)')
else:
    print(f'⚠️ Below target. Gap: {(TARGET_ACCURACY - best_accuracy)*100:.2f}%')

In [ ]:
top_models = results_df.head(3)['Model'].tolist()
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for idx, model_name in enumerate(top_models):
    y_pred = predictions[model_name]
    cm = confusion_matrix(y_test, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[idx], cbar=False)
    acc = accuracy_score(y_test, y_pred)
    axes[idx].set_title(f'{model_name}\n{acc:.4f}')
    axes[idx].set_xlabel('Predicted')
    axes[idx].set_ylabel('Actual')

plt.tight_layout()
plt.savefig('output/confusion_matrices.png', dpi=300, bbox_inches='tight')
plt.show()
print('✅ Confusion matrices displayed')

## 7. CONCLUSION

In [ ]:
print('='*60)
print('FINAL RESULTS SUMMARY')
print('='*60)
print(f'\nBest Model: {best_model_name}')
print(f'Accuracy: {best_accuracy:.4f} ({best_accuracy*100:.2f}%)')
print(f'Target: {TARGET_ACCURACY*100}%')
print(f'Status: {"✅ TARGET ACHIEVED" if best_accuracy >= TARGET_ACCURACY else "⚠️ Below Target"}')
print(f'\nTop 3 Models:')
for i, row in results_df.head(3).iterrows():
    print(f'  {row["Model"]}: {row["Accuracy"]:.4f}')

if best_accuracy < TARGET_ACCURACY:
    gap = (TARGET_ACCURACY - best_accuracy) * 100
    print(f'\nGap to Target: {gap:.2f}%')
    print(f'\nRecommendations for Improvement:')
    print(f'1. Collect more data (especially minority class)')
    print(f'2. Try ensemble methods or stacking')
    print(f'3. Engineer additional features')
    print(f'4. Perform more aggressive hyperparameter tuning')
    print(f'5. Experiment with different scaling/preprocessing')
else:
    print(f'\n✅ SUCCESS! Model achieved {best_accuracy*100:.2f}% accuracy')
    print(f'\nNext Steps:')
    print(f'1. Deploy model to production')
    print(f'2. Monitor performance on new data')
    print(f'3. Retrain periodically with new samples')

print('\n' + '='*60)
print(f'Pipeline completed at: {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}')
print('='*60)